# Data cleaning

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_excel("uci_concrete_data.xlsx",sheet_name="Sheet1")
df.describe()

In [ ]:
df.isnull().sum()

### SimpleImputer

In [ ]:

imp = SimpleImputer(missing_values=np.nan, strategy='mean')
imp.fit(df)
df_sim_imp = pd.DataFrame(imp.transform(df), columns=df.columns)
#print(df_sim_imp)
#df_sim_imp.isnull().sum()

### Nearest neighbors imputation

In [ ]:
imp_near = KNNImputer(n_neighbors=5, weights='distance')
df_near = pd.DataFrame(imp_near.fit_transform(df), columns=df.columns)
#df_near.isnull().sum()
#df_near

### Dropping the rows

In [ ]:
df_drop = df.dropna()

#df_drop.isnull().sum()

df_drop.describe()
#df.isnull().sum()
#df.describe()

### Drop duplicates

In [ ]:
df_drop.duplicated().sum()
df_drop_dup = df_drop.drop_duplicates()
df_drop_dup.duplicated().sum()

### Correlation with respect to compressive strength

In [ ]:
df_drop_dup.corr() #Pearson correlation
# we look to the highest positive and the lowest negative ratio wwith respect to compressive strength

### Visualization

In [ ]:
df_drop_dup.boxplot(rot=90, figsize=(12,8), fontsize=6)
plt.show()

### Differentiate input and output data

In [ ]:
X = df.iloc[:, 0:8]
Y = df.iloc[:, 8]
# X = df_drop_dup.iloc[:, 0:8]
# Y = df_drop_dup.iloc[:, 8]

### Common seed

In [ ]:
from sklearn.model_selection import train_test_split, KFold

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
kf = KFold(n_splits=3, random_state=42, shuffle=True)

# Boosting 


## Raw data without cleaning

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBRegressor


### Histogram Gradient Boosting

In [ ]:
# hist = HistGradientBoostingRegressor(random_state=42)
# gridSe = GridSearchCV(estimator=hist, param_grid={'max_iter': (50, 100, 200, 300, 500, 1000), 'max_depth': (3,4,5,10,20,50,100,None)}, cv=kf)

#### Fit a model

In [ ]:
# gridSe_fit = gridSe.fit(X_train, Y_train)
# gridSe_fit_best_params = gridSe_fit.best_params_
# print(gridSe_fit_best_params)

#### Prediction

In [ ]:
# hist_Y_pred = gridSe_fit.predict(X_test)

#### Calculation of Metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [ ]:
# hist_mae = mean_absolute_error(Y_test, hist_Y_pred)
# hist_r2 = r2_score(Y_test, hist_Y_pred)
# hist_rmse = root_mean_squared_error(Y_test,hist_Y_pred)
# hist_residual = np.subtract(Y_test, hist_Y_pred)

In [ ]:
# metrics_dict = {"MAE":[hist_mae], "RMSE":[hist_rmse], "R":[hist_r2]}
# hist_metrics = pd.DataFrame(metrics_dict)
# hist_metrics

In [ ]:
# plt.scatter(hist_Y_pred, hist_residual, c="m")
# plt.axhline(0)
# plt.figure()
# plt.show()

### Comparison beteween XGBRegressor and HistGradientBoostingRegressor

In [ ]:
models = {'XGBRegressor': XGBRegressor(random_state=42), 
          'HistGradientBoostingRegressor': HistGradientBoostingRegressor(random_state=42)}
params = {'XGBRegressor':{'n_estimators': (50, 100, 200, 300, 500, 1000), 'max_depth': (3,4,5,10,20,50,100,None)},
          'HistGradientBoostingRegressor':{'max_iter': (50, 100, 200, 300, 500, 1000), 'max_depth': (3,4,5,10,20,50,100,None)}}


# XGBoost evaluates the Gain for each possible split and selects the best one.
# HistGradientBoostingRegressor splits the data to bins (255 by default)

In [ ]:
grid_loop = {}
grid_loop_best_params = {}
Y_pred = {}
mae = {}
rmse = {}
r2 = {}
residual = {}
for name, model in models.items():
    grid_loop[name] = GridSearchCV(estimator=model, param_grid=params[name], cv=kf)
    grid_loop[name].fit(X_train, Y_train)
    grid_loop_best_params[name] = grid_loop[name].best_params_
    Y_pred[name] = grid_loop[name].predict(X_test)
    mae[name] = mean_absolute_error(Y_test, Y_pred[name])
    rmse[name] = root_mean_squared_error(Y_test,Y_pred[name])
    r2[name] = r2_score(Y_test, Y_pred[name])
    residual[name] = np.subtract(Y_test, Y_pred[name])

In [ ]:
metrics_dict = {"MAE":mae, "RMSE":rmse, "R":r2, "BestParams": grid_loop_best_params}
metrics = pd.DataFrame(metrics_dict)
params_df=metrics['BestParams'].apply(pd.Series)
metrics = metrics.drop(columns=['BestParams'])
final_metrics = pd.concat([metrics, params_df], axis=1)

final_metrics

In [ ]:
fig, axes = plt.subplots(1, 2)
axes[0].scatter(Y_pred['XGBRegressor'], residual['XGBRegressor'], c="m")
axes[1].scatter(Y_pred['HistGradientBoostingRegressor'], residual['HistGradientBoostingRegressor'], c="m")
axes[0].axhline(0)
axes[1].axhline(0)
plt.show()

## Boosting models after data cleaning

In [ ]:
X_c = df_drop_dup.iloc[:, 0:8]
Y_c = df_drop_dup.iloc[:, 8]

In [ ]:
X_train_c, X_test_c, Y_train_c, Y_test_c = train_test_split(X_c, Y_c, test_size=0.2, random_state=42)
kf_c = KFold(n_splits=3, random_state=42, shuffle=True)

In [ ]:
grid_loop_c = {}
grid_loop_best_params_c = {}
Y_pred_c = {}
mae_c = {}
rmse_c = {}
r2_c = {}
residual_c = {}
for name, model in models.items():
    grid_loop_c[name] = GridSearchCV(estimator=model, param_grid=params[name], cv=kf_c)
    grid_loop_c[name].fit(X_train_c, Y_train_c)
    grid_loop_best_params_c[name] = grid_loop_c[name].best_params_
    Y_pred_c[name] = grid_loop_c[name].predict(X_test_c)
    mae_c[name] = mean_absolute_error(Y_test_c, Y_pred_c[name])
    rmse_c[name] = root_mean_squared_error(Y_test_c,Y_pred_c[name])
    r2_c[name] = r2_score(Y_test_c, Y_pred_c[name])
    residual_c[name] = np.subtract(Y_test_c, Y_pred_c[name])

In [ ]:
metrics_dict_c = {"MAE_CD":mae_c, "RMSE_CD":rmse_c, "R_CD":r2_c, "BestParams_CD": grid_loop_best_params_c}
metrics_c = pd.DataFrame(metrics_dict_c)
params_df_c=metrics_c['BestParams_CD'].apply(pd.Series)
metrics_c = metrics_c.drop(columns=['BestParams_CD'])
final_metrics_c = pd.concat([metrics_c, params_df_c], axis=1)

In [ ]:
fig, axes = plt.subplots(1, 2)
axes[0].scatter(Y_pred_c['XGBRegressor'], residual_c['XGBRegressor'], c="m")
axes[1].scatter(Y_pred_c['HistGradientBoostingRegressor'], residual_c['HistGradientBoostingRegressor'], c="m")
axes[0].axhline(0)
axes[1].axhline(0)
plt.show()

In [ ]:
final_metrics

In [ ]:
final_metrics_c